# Paper 1 - eating behavior

### Define the relevant directories used in this paper - RUN EVERY TIME

Data is pulled from the standardized data folder; subsequently, it is stored and managed in the paper 1 folder. 

In [17]:
import os

# Define the source and output directories
source_directory = r"C:\Users\Felhasználó\Desktop\Projects\PNK_DB2\DB2_standard"
wgcauses_directory = r"C:\Users\Felhasználó\Desktop\Projects\PNK_DB2\paper1_emotional\wg_causes"

# Ensure the output directory exists
os.makedirs(wgcauses_directory, exist_ok=True)

# 1. Data preparation

### Create an input table for survival analysis

In [18]:
import os
import sqlite3
import pandas as pd
import numpy as np
from datetime import timedelta

"""
1. CONFIGURATION
"""

# 1a. Define I/O directories, used databases and tables in them
input_db  = os.path.join(source_directory, "pnk_db2_filtered.sqlite")
output_db = os.path.join(wgcauses_directory, "survival_analysis.sqlite")

input_measurements    = "measurements_filtered"
input_medical_records = "medical_records_filtered"
output_table          = "sa_input_table"

# 1b. Define analysis parameters: weight loss targets, time windows, and window span
weight_loss_targets = [5, 10, 15]     # % thresholds
time_windows        = [40, 60, 80]    # days
window_span         = 10              # +/- days around each window

# 1c. Define which variables to fetch from medical records
fetch_from_records = [
  "patient_id", "medical_record_id",
  "dietitian_visits", 
  "sex_f", "age_when_creating_record AS age", # for clarity in downstream operations and a more present, rename 'age_when_creating_record' to 'age'
  "height_m",
  "hunger_yn", "satiety_yn", "emotional_eating_yn",
  "emotional_eating_value_likert",
  "quantity_control_likert", "impulse_control_likert",
  "weight_gain_cause", "genomics_sample_id"
]

# 1d. Build output column order dynamically
# First come the columns with metadata and overall followup info
# after the followup summary columns, we will inject fixed‐time & time‐to‐event columns
# after these, the columns containing predictor and confounder columns (clinical and eating behavior data) are inserted
followup_columns = [
    "patient_id","medical_record_id",
    "baseline_date","last_aval_date","total_followup_days",
    "dietitian_visits","nr_total_measurements","avg_days_between_measurements",
    "baseline_weight_kg","last_aval_weight_kg","total_wl_kg","total_wl_%", 
    "baseline_bmi","final_bmi","bmi_reduction", "baseline_fat_%","last_aval_fat_%","total_fat_loss_%",
    "baseline_muscle_%","last_aval_muscle_%","total_muscle_change_%"
]
# -> dynamically insert fixed timepoint and time-to-event columns here  
predictor_columns = ["sex_f","age","height_m","hunger_yn","satiety_yn","emotional_eating_yn",
                "emotional_eating_value_likert","quantity_control_likert","impulse_control_likert",
                "weight_gain_cause","genomics_sample_id"]
# Use a function for inserting dynamic columns where appropriate - between followup and predictor columns
def make_column_order():
    cols = followup_columns.copy()
    #  fixed‐time
    for w in time_windows:
        prefix = f"{w}d"
        cols += [
            f"{prefix}_weight_kg",
            f"wl_{prefix}_kg",
            f"wl_{prefix}_%",
            f"{prefix}_fat_%",
            f"{prefix}_fat_loss_%",
            f"{prefix}_muscle_%",
            f"{prefix}_muscle_change_%",
            f"{prefix}_date",
            f"days_to_{prefix}_measurement",
            f"{prefix}_dropout"
        ]
    # time‐to‐event
    for t in weight_loss_targets:
        prefix = f"{t}%_wl"
        cols += [
            f"{prefix}_achieved",
            f"{prefix}_%",
            f"{prefix}_date",
            f"days_to_{prefix}"
        ]
    cols += predictor_columns
    return cols
output_column_order = make_column_order()

"""
2. DATA LOADING
"""
# Load measurements and medical records tables from the input database
def load_measurements(conn) -> pd.DataFrame:
    df = pd.read_sql(f"SELECT * FROM {input_measurements}", conn,
                     parse_dates=["measurement_date"])
    df = df.sort_values(
        ["patient_id","medical_record_id","measurement_date"]
    )
    return df
def load_med_records(conn) -> pd.DataFrame:
    df = pd.read_sql(f"SELECT {','.join(fetch_from_records)} FROM {input_medical_records}", conn)
    return df

# BASELINE & MERGE

def extract_baseline(meas: pd.DataFrame) -> pd.DataFrame:
    # assume `first_in_record` flags the baseline row per record
    base = meas[meas["first_in_record"]==1].copy()
    base = base.rename(columns={
        "measurement_date":"baseline_date",
        "weight_kg":"baseline_weight_kg",
        "bmi":"baseline_bmi",
        "fat_%":"baseline_fat_%",
        "muscle_%":"baseline_muscle_%"
    })
    return base[[
        "patient_id","medical_record_id",
        "baseline_date","baseline_weight_kg","baseline_bmi",
        "baseline_fat_%","baseline_muscle_%"
    ]]

def merge_baseline_and_records(baseline, recs):
    df = baseline.merge(
        recs,
        on=["patient_id","medical_record_id"],
        how="left"
    )
    return df

# CALCULATIONS

def calc_overall_followup(meas, row):
    pid, mrid = row["patient_id"], row["medical_record_id"]
    baseline_date = row["baseline_date"]
    group = meas.query(
      "patient_id==@pid & medical_record_id==@mrid & measurement_date> @baseline_date"
    ).sort_values("measurement_date")
    if len(group)==0:
        last = row.copy()
        last["last_aval_date"] = row["baseline_date"]
        last["total_followup_days"] = 1
        last["last_aval_weight_kg"] = row["baseline_weight_kg"]
        last["total_wl_kg"], last["total_wl_%"] = 0,0
        last["final_bmi"] = row["baseline_bmi"]
        last["bmi_reduction"] = 0
        last["last_aval_fat_%"]      = row["baseline_fat_%"]
        last["last_aval_muscle_%"]   = row["baseline_muscle_%"]
        last["total_fat_loss_%"]     = 0
        last["total_muscle_change_%"]= 0
    else:
        last = row.copy()
        last_meas = group.iloc[-1]
        dt = (last_meas.measurement_date - baseline_date).days + 1
        last["last_aval_date"] = last_meas.measurement_date
        last["total_followup_days"] = dt
        last["last_aval_weight_kg"] = last_meas.weight_kg
        last["total_wl_kg"] = last_meas.weight_kg - row["baseline_weight_kg"]
        last["total_wl_%"] = 100 * last["total_wl_kg"] / row["baseline_weight_kg"]
        last["final_bmi"] = last_meas.bmi
        last["bmi_reduction"] = last_meas.bmi - row["baseline_bmi"]
        last["last_aval_fat_%"]  = last_meas["fat_%"]
        last["total_fat_loss_%"] = last_meas["fat_%"] - row["baseline_fat_%"]
        last["last_aval_muscle_%"]   = last_meas["muscle_%"]
        last["total_muscle_change_%"] = last_meas["muscle_%"] - row["baseline_muscle_%"]
    # visits & counts
    subset = meas.query(
      "patient_id==@pid & medical_record_id==@mrid"
    )
    last["nr_total_measurements"] = len(subset)
    # REMOVE last["dietitian_visits"] = row["dietitian_visits"]
    if last["nr_total_measurements"]>1:
        last["avg_days_between_measurements"] = \
          (last["total_followup_days"] - 1)/(last["nr_total_measurements"]-1)
    else:
        last["avg_days_between_measurements"] = np.nan

    return last

def calc_fixed_timepoints(meas, row):
    out = {}
    pid, mrid = row["patient_id"], row["medical_record_id"]
    baseline = row["baseline_date"]
    for w in time_windows:
        lo = baseline + timedelta(days=w-window_span)
        hi = baseline + timedelta(days=w+window_span)
        window_meas = meas.query(
          "patient_id==@pid & medical_record_id==@mrid"
          " & measurement_date>=@lo & measurement_date<=@hi"
        ).copy()
        prefix = f"{w}d"
        if window_meas.empty:
            out[f"{prefix}_dropout"] = 1
            out[f"{prefix}_weight_kg"] = np.nan
            out[f"wl_{prefix}_kg"]   = np.nan
            out[f"wl_{prefix}_%"]    = np.nan
            out[f"{prefix}_fat_%"]         = np.nan
            out[f"{prefix}_fat_loss_%"]    = np.nan
            out[f"{prefix}_muscle_%"]      = np.nan
            out[f"{prefix}_muscle_change_%"]= np.nan
            out[f"{prefix}_date"]    = pd.NaT
            out[f"days_to_{prefix}_measurement"] = np.nan
        else:
            window_meas["dist"] = (window_meas.measurement_date - baseline).abs().dt.days
            take = window_meas.loc[window_meas.dist.idxmin()]
            out[f"{prefix}_dropout"] = 0
            out[f"{prefix}_weight_kg"] = take.weight_kg
            out[f"wl_{prefix}_kg"] = take.weight_kg - row["baseline_weight_kg"]
            out[f"wl_{prefix}_%"]  = 100 * out[f"wl_{prefix}_kg"] / row["baseline_weight_kg"]
            out[f"{prefix}_fat_%"]         = take["fat_%"]
            out[f"{prefix}_fat_loss_%"] = take["fat_%"] - row["baseline_fat_%"]
            out[f"{prefix}_muscle_%"]      = take["muscle_%"]
            out[f"{prefix}_muscle_change_%"]  = take["muscle_%"] - row["baseline_muscle_%"]
            out[f"{prefix}_date"] = take.measurement_date
            out[f"days_to_{prefix}_measurement"] = (take.measurement_date - baseline).days + 1
    return out

def calc_time_to_targets(meas, row):
    out = {}
    pid, mrid = row["patient_id"], row["medical_record_id"]
    baseline = row["baseline_date"]
    group = meas.query(
      "patient_id==@pid & medical_record_id==@mrid"
      " & measurement_date> @baseline"
    ).sort_values("measurement_date")
    for t in weight_loss_targets:
        achieved = False
        for _, r in group.iterrows():
            wl_pct = 100*(row["baseline_weight_kg"]-r.weight_kg)/row["baseline_weight_kg"]
            if wl_pct>=t:
                achieved = True
                out[f"{t}%_wl_achieved"] = 1
                out[f"{t}%_wl_%"] = wl_pct
                out[f"{t}%_wl_date"] = r.measurement_date
                out[f"days_to_{t}%_wl"] = (r.measurement_date - baseline).days + 1
                break
        if not achieved:
            out[f"{t}%_wl_achieved"] = 0
            out[f"{t}%_wl_%"] = np.nan
            out[f"{t}%_wl_date"] = pd.NaT
            out[f"days_to_{t}%_wl"] = np.nan
    return out


# ORCHESTRATION

def build_sa_input_table():
    conn_in  = sqlite3.connect(input_db)
    # load
    meas   = load_measurements(conn_in)
    recs   = load_med_records(conn_in)
    conn_in.close()

    # baseline + merge
    base    = extract_baseline(meas)
    merged  = merge_baseline_and_records(base, recs)

    # iterate each patient‐record
    rows = []
    for _, row in merged.iterrows():
        # 1) overall follow‐up
        overall_followup = calc_overall_followup(meas, row)
        out_dict = overall_followup.to_dict()
        # 2) fixed timepoints
        fixed_timepoints = calc_fixed_timepoints(meas, row)
        out_dict.update(fixed_timepoints)
        # 3) time-to-event
        time_to_event = calc_time_to_targets(meas, row)
        out_dict.update(time_to_event)
        rows.append(out_dict)

    df_out = pd.DataFrame(rows)
    # reorder
    df_out = df_out.reindex(columns=output_column_order)

    # save
    conn_out = sqlite3.connect(output_db)
    df_out.to_sql(output_table, conn_out, if_exists="replace", index=False)
    conn_out.close()
    print(f"Saved {len(df_out)} rows to {output_db}::{output_table}")

if __name__=="__main__":
    build_sa_input_table()


KeyboardInterrupt: 

# 2. Data analysis

## Correlations

### Baseline population and outcome-level summary statistics

#### description

This code builds and saves two summary tables—population‐level and outcome‐level—from the survival‐analysis input table in survival_analysis.sqlite. It reads the wide sa_input_table, computes descriptive statistics for baseline and follow‐up metrics, then writes the results back as population_summary and outcome_summary.

**Configuration**

*   Define `paper1_directory`, then set paths and table names.
*   Specify analysis parameters:
    *   `weight_loss_targets = [5, 10, 15]`
    *   `time_windows = [40, 60, 80]`
*   Configure row definitions:
    *   `pop_summary_row_config`: maps display names → (`column`, `type`) for baseline variables
    *   `output_summary_row_config`: maps follow‐up metrics → (`column`, `type`), then extended dynamically for each window (`wl_{w}d`) and target (`{t}%_wl`)

**Helper Functions**

*   `fmt_mean_sd(s)`: “mean ± SD” for numeric series
*   `fmt_median_iqr(s)`: “median (Q1–Q3)”
*   `fmt_min_max(s)`: “min–max”
*   `fmt_n_perc(n, total)`: “N (X.X%)” for counts

**Summary Functions**

*   `generate_population_summary(df)`:
    *   Iterates over `pop_summary_row_config`
    *   For each variable, computes N(%), mean±SD, median(IQR), min–max as applicable
*   `generate_outcome_summary(df, tgt, tws)`:
    *   Iterates over `output_summary_row_config`
    *   Summarizes overall follow‐up, fixed‐window weight‐loss, dropout flags, and time‐to‐target achievements

**Orchestration**

*   `create_and_save_summary_tables()`:
    *   Connect to `db_path` and load `sa_input_table` into a DataFrame
    *   Call `generate_population_summary(df)` → `pop_sum`
    *   Call `generate_outcome_summary(df, weight_loss_targets, time_windows)` → `outcome_sum`
    *   Display both tables in the notebook
    *   Save `pop_sum` → table `population_summary` and `outcome_sum` → table `outcome_summary`

**Execution & Summary**

*   When run as a script (`if __name__=="__main__":`), it invokes `create_and_save_summary_tables()` and prints “✔︎ Summaries saved.”

#### code

In [ ]:
import os
import sqlite3
import pandas as pd
import numpy as np

# 1) CONFIGURATION --------------------------------------------------

# Make sure you set this before running:
# paper1_directory = r"C:\Users\Felhasználó\Desktop\Projects\PNK_DB2\paper1_emotional"
db_path       = os.path.join(paper1_directory, "survival_analysis.sqlite")
input_table   = "sa_input_table"
pop_table     = "population_summary"
outcome_table = "outcome_summary"

# These should match your SA input‐table construction
weight_loss_targets = [5, 10, 15]   # percent WL
time_windows        = [40, 60, 80]  # days

# ROW CONFIGURATIONS --------------------------------------------------
# Each entry: display_name → (column_name, type)
# type ∈ {"numeric", "boolean", "availability"}
pop_summary_row_config = {
    "Total population":    (None,              "constant"),
    "Women":               ("sex_f",           "boolean"),
    "Age":                 ("age",              "numeric"),
    "Height (m)":          ("height_m",         "numeric"),
    "Baseline weight (kg)":("baseline_weight_kg","numeric"),
    "Last available weight (kg)":("last_aval_weight_kg","numeric"),
    "Baseline BMI (kg/m^2)": ("baseline_bmi",   "numeric"),
    "Last available BMI (kg/m^2)":("final_bmi",   "numeric"),
    "Baseline fat mass (%)":     ("baseline_fat_%",     "numeric"),
    "Last available fat mass (%)":("last_aval_fat_%",   "numeric"),
    "Baseline muscle mass (%)":  ("baseline_muscle_%",  "numeric"),
    "Last available muscle mass (%)":("last_aval_muscle_%","numeric"),
    "Emotional eating (1-10)":("emotional_eating_value_likert","numeric"),
    "Quantity control (1-10)":("quantity_control_likert",   "numeric"),
    "Impulse control (1-10)": ("impulse_control_likert",    "numeric"),
    "Hunger":              ("hunger_yn",       "boolean"),
    "Satiety":             ("satiety_yn",      "boolean"),
    "Emotional eating":    ("emotional_eating_yn","boolean"),
    "Weight gain cause available":("weight_gain_cause","availability"),
    "Genomics available":  ("genomics_sample_id","availability"),
}

output_summary_row_config = {
    "Total population":     (None,               "constant"),
    "Total followup (days)":("total_followup_days","numeric"),
    "Avg days between visits":("avg_days_between_measurements","numeric"),
    "Total WL (kg)":        ("total_wl_kg",      "numeric"),
    "Total WL (%)":         ("total_wl_%",       "numeric"),
    "BMI reduction":        ("bmi_reduction",    "numeric"),
    "Total fat loss (%)":         ("total_fat_loss_%",       "numeric"),
    "Total muscle mass change (%)":("total_muscle_change_%",  "numeric"),
    "Instant dropouts":     ("total_followup_days","dropout_instant"),
    # fixed windows and targets will be appended programmatically below...
}

# After these you can dynamically extend output_summary_row_config for each time window and each %
for w in time_windows:
    output_summary_row_config[f"WL at {w}d (kg)"] = (f"wl_{w}d_kg", "numeric")
    output_summary_row_config[f"WL at {w}d (%)"]  = (f"wl_{w}d_%",  "numeric")
    output_summary_row_config[f"Fat loss at {w}d (%)"]       = (f"{w}d_fat_loss_%",      "numeric")
    output_summary_row_config[f"Muscle mass change at {w}d (%)"] = (f"{w}d_muscle_change_%","numeric")
    output_summary_row_config[f"Completers at {w}d"] = (f"{w}d_dropout", "dropout_flag")
for t in weight_loss_targets:
    output_summary_row_config[f"Achieved {t}% WL"]   = (f"{t}%_wl_achieved","boolean")
    output_summary_row_config[f"Days to {t}% WL"]     = (f"days_to_{t}%_wl",   "numeric")

# Now rewrite your generate_population_summary to loop over POP_ROW_CONFIG,
# and generate_outcome_summary to loop over output_summary_row_config.

# 2) HELPER FUNCTIONS ----------------------------------------------

def fmt_mean_sd(s: pd.Series, dec: int = 1) -> str:
    """Return 'mean ± SD' for a numeric series or NaN if too few data."""
    s = pd.to_numeric(s, errors="coerce").dropna()
    if len(s) < 2:
        return np.nan
    m, sd = s.mean(), s.std()
    return f"{m:.{dec}f} ± {sd:.{dec}f}"

def fmt_median_iqr(s: pd.Series, dec: int = 1) -> str:
    """Return 'median (Q1–Q3)' for a numeric series or NaN."""
    s = pd.to_numeric(s, errors="coerce").dropna()
    if len(s) == 0:
        return np.nan
    q1, med, q3 = np.percentile(s, [25, 50, 75])
    return f"{med:.{dec}f} ({q1:.{dec}f}–{q3:.{dec}f})"

def fmt_min_max(s: pd.Series, dec: int = 1) -> str:
    """Return 'min–max' for a numeric series or NaN."""
    s = pd.to_numeric(s, errors="coerce").dropna()
    if len(s) == 0:
        return np.nan
    return f"{s.min():.{dec}f}–{s.max():.{dec}f}"

def fmt_n_perc(count: int, total: int, dec: int = 1) -> str:
    """Format N and percent of total as 'N (X.X%)'."""
    if total == 0:
        return "0 (0.0%)"
    pct = 100 * count / total
    return f"{count} ({pct:.{dec}f}%)"

# 3) SUMMARY FUNCTIONS --------------------------------------------

def generate_population_summary(df: pd.DataFrame) -> pd.DataFrame:
    """
    Returns a DataFrame indexed by Variable with columns:
      'N (%)', 'Mean (SD)', 'Median (IQR)', 'Min–Max'
    """
    total_n = len(df)
    print(f"Generating population summary (N={total_n})...")
    rows = []

    for display_name, (col, kind) in pop_summary_row_config.items():
        if kind=="constant":
            n = total_n
            pct = 100
            rows.append({ "Variable": display_name,
                        "N (%)": fmt_n_perc(n, total_n),
                        "Mean (SD)": np.nan,
                        "Median (IQR)": np.nan,
                        "Min–Max": np.nan })
        elif kind=="numeric":
            s = df[col]
            rows.append({ "Variable": display_name,
                        "N (%)": fmt_n_perc(s.notna().sum(), total_n),
                        "Mean (SD)": fmt_mean_sd(s),
                        "Median (IQR)": fmt_median_iqr(s),
                        "Min–Max": fmt_min_max(s) })
        elif kind=="boolean":
            n = df[col].eq(1).sum()
            rows.append({ "Variable": display_name,
                        "N (%)": fmt_n_perc(n, total_n),
                        "Mean (SD)": np.nan,
                        "Median (IQR)": np.nan,
                        "Min–Max": np.nan })
        elif kind=="availability":
            n = df[col].notna().sum()
            rows.append({ "Variable": display_name,
                        "N (%)": fmt_n_perc(n, total_n),
                        "Mean (SD)": np.nan,
                        "Median (IQR)": np.nan,
                        "Min–Max": np.nan })

    pop_df = pd.DataFrame(rows).set_index("Variable")
    return pop_df

def generate_outcome_summary(df: pd.DataFrame,
                             tgt: list[int],
                             tws: list[int]) -> pd.DataFrame:
    """
    Returns a DataFrame indexed by Variable with columns:
      'N (%)', 'Mean (SD)', 'Median (IQR)', 'Min–Max'
    Summarizes overall, fixed windows, instantaneous dropout & time-to-targets.
    """
    total_n = len(df)
    print(f"Generating outcome summary (N={total_n})...")
    rows = []

    # … inside generate_outcome_summary, after you compute total_n …
    rows = []
    print(f"Generating outcome summary (N={total_n})…")
    
    for display_name, (col, kind) in output_summary_row_config.items():
        # constant rows (Total population)
        if kind == "constant":
            rows.append({
                "Variable": display_name,
                "N (%)": fmt_n_perc(total_n, total_n),
                "Mean (SD)": np.nan,
                "Median (IQR)": np.nan,
                "Min–Max": np.nan
            })
    
        # pure numeric summaries
        elif kind == "numeric":
            if col in df:
                s = df[col]
                rows.append({
                    "Variable": display_name,
                    "N (%)": fmt_n_perc(s.notna().sum(), total_n),
                    "Mean (SD)": fmt_mean_sd(s),
                    "Median (IQR)": fmt_median_iqr(s),
                    "Min–Max": fmt_min_max(s)
                })
            else:
                print(f"  Warning: '{col}' not found for {display_name}")
    
        # simple boolean flags (e.g. Achieved 5% WL)
        elif kind == "boolean":
            if col in df:
                n_yes = df[col].eq(1).sum()
                rows.append({
                    "Variable": display_name,
                    "N (%)": fmt_n_perc(n_yes, total_n),
                    "Mean (SD)": np.nan,
                    "Median (IQR)": np.nan,
                    "Min–Max": np.nan
                })
            else:
                print(f"  Warning: '{col}' not found for {display_name}")
    
        # instant dropouts: total_followup_days == 1
        elif kind == "dropout_instant":
            if col in df:
                n_drop = df[col].eq(1).sum()
                rows.append({
                    "Variable": display_name,
                    "N (%)": fmt_n_perc(n_drop, total_n),
                    "Mean (SD)": np.nan,
                    "Median (IQR)": np.nan,
                    "Min–Max": np.nan
                })
            else:
                print(f"  Warning: '{col}' not found for {display_name}")
    
        # completers at fixed windows: dropout_flag means 1‐dropped, 0‐completer
        elif kind == "dropout_flag":
            if col in df:
                n_comp = (df[col] == 0).sum()
                rows.append({
                    "Variable": display_name,
                    "N (%)": fmt_n_perc(n_comp, total_n),
                    "Mean (SD)": np.nan,
                    "Median (IQR)": np.nan,
                    "Min–Max": np.nan
                })
            else:
                print(f"  Warning: '{col}' not found for {display_name}")
    
        else:
            # catch‐all for anything else you might add
            print(f"  Warning: Unknown kind='{kind}' for {display_name}")
            
    # finally…
    out_df = pd.DataFrame(rows).set_index("Variable")
    return out_df

# 4) ORCHESTRATION FUNCTION ----------------------------------------

def create_and_save_summary_tables():
    print(f"\nConnecting to DB: {db_path}")
    if not os.path.exists(db_path):
        print("ERROR: DB not found.")
        return

    # load SA input
    with sqlite3.connect(db_path) as conn:
        print(f"Reading table: {input_table}")
        df = pd.read_sql_query(f"SELECT * FROM {input_table}", conn)

    if df.empty:
        print("ERROR: Input table is empty.")
        return

    # generate
    pop_sum    = generate_population_summary(df)
    outcome_sum= generate_outcome_summary(df, weight_loss_targets, time_windows)
    # Check the tables within the notebook
    print("\nPopulation summary:"); display(pop_sum)
    print("\nOutcome summary:");    display(outcome_sum)

    # save back
    with sqlite3.connect(db_path) as conn:
        print(f"Saving population summary to {pop_table}")
        pop_sum.to_sql(pop_table, conn, if_exists="replace", index=True)
        print(f"Saving outcome summary to {outcome_table}")
        outcome_sum.to_sql(outcome_table, conn, if_exists="replace", index=True)

    print("✔︎ Summaries saved.")

# 5) EXECUTE --------------------------------------------------------

if __name__ == "__main__":
    create_and_save_summary_tables()


Connecting to DB: C:\Users\Felhasználó\Desktop\Projects\PNK_DB2\paper1_emotional\survival_analysis.sqlite
Reading table: sa_input_table
Generating population summary (N=1171)...
Generating outcome summary (N=1171)...
Generating outcome summary (N=1171)…

Population summary:


,N (%),Mean (SD),Median (IQR),Min–Max
Variable,,,,
Total population,1171 (100.0%),NaN,NaN,NaN
Women,933 (79.7%),NaN,NaN,NaN
Age,1171 (100.0%),47.0 ± 10.4,47.0 (41.0–54.0),18.0–83.0
Height (m),1171 (100.0%),1.7 ± 0.1,1.6 (1.6–1.7),1.4–1.9
Baseline weight (kg),1171 (100.0%),85.9 ± 11.4,84.5 (78.2–91.2),57.6–131.2
Last available weight (kg),1171 (100.0%),77.6 ± 11.6,76.6 (69.2–83.8),50.7–130.5
Baseline BMI (kg/m^2),1171 (100.0%),31.1 ± 2.5,31.2 (29.1–32.9),27.0–40.2
Last available BMI (kg/m^2),1171 (100.0%),28.1 ± 3.0,27.8 (25.9–30.2),20.8–38.4
Baseline fat mass (%),1171 (100.0%),42.5 ± 5.2,42.9 (38.8–47.6),25.5–50.0



Outcome summary:


,N (%),Mean (SD),Median (IQR),Min–Max
Variable,,,,
Total population,1171 (100.0%),NaN,NaN,NaN
Total followup (days),1171 (100.0%),85.4 ± 76.6,65.0 (28.0–122.0),1.0–545.0
Avg days between visits,1117 (95.4%),7.1 ± 5.7,6.3 (3.6–8.6),0.0–70.0
Total WL (kg),1171 (100.0%),-8.3 ± 5.9,-7.6 (-11.8–-3.6),-29.4–7.1
Total WL (%),1171 (100.0%),-9.6 ± 6.6,-9.0 (-13.8–-4.4),-32.1–9.2
BMI reduction,1171 (100.0%),-3.0 ± 2.1,-2.8 (-4.2–-1.4),-11.4–2.9
Total fat loss (%),1171 (100.0%),-5.6 ± 4.0,-5.1 (-8.0–-2.5),-19.5–4.7
Total muscle mass change (%),1171 (100.0%),3.2 ± 2.4,2.9 (1.4–4.7),-2.5–12.3
Instant dropouts,59 (5.0%),NaN,NaN,NaN


Saving population summary to population_summary
Saving outcome summary to outcome_summary
✔︎ Summaries saved.


### Comparative summary stats

##### Explanation of the first draft pipeline

##### Code of the first draft pipeline

##### corr w wg causes

In [20]:
import os
import sqlite3
import pandas as pd
import numpy as np
from scipy import stats
import warnings

# --- GLOBAL DIRECTORY DEFINITIONS (User needs to set these) ---
# These were used in your DB_PATH definitions but not defined in the script snippet.
# Please ensure these point to the correct locations in your environment.
source_directory = r"C:\Users\Felhasználó\Desktop\Projects\PNK_DB2\DB2_standard"
wgcauses_directory = r"C:\Users\Felhasználó\Desktop\Projects\PNK_DB2\paper1_emotional\wg_causes"

# --- CONFIGURATION ---
DB_PATH_SURVIVAL = os.path.join(wgcauses_directory, "survival_analysis.sqlite")
DB_PATH_EMOTIONAL = os.path.join(source_directory, "pnk_db2_filtered.sqlite")
SA_INPUT_TABLE = "sa_input_table" # Table with outcome variables
MEDICAL_RECORDS_TABLE = "medical_records_filtered" # Table with patient_id and cause columns
OUTPUT_TABLE = "correlation_by_cause_summary_v2" # Updated output table name

# Weight gain cause columns from medical_records_filtered
WEIGHT_GAIN_CAUSE_COLUMNS = [
    'womens_health_and_pregnancy',
    'mental_health',
    'family_issues',
    'medication_disease_injury',
    'physical_inactivity',
    'eating_habits',
    'schedule',
    'smoking_cessation',
    'post_treatment_weight_regain',
    'pandemic',
    'habits_and_circumstances',
    'none_of_above',
]

# Analysis parameters
time_windows = [40, 60, 80]
weight_loss_targets = [5, 10, 15]

# --- DYNAMIC ROWS CONFIG (Outcomes/Variables to analyze) ---
rows_config = {
    "N": {"type": "categorical", "format": "n_count"},
    "age": {"type": "continuous", "format": "mean_sd"},
    "sex_f": {"type": "categorical", "format": "n_perc"}, # Binary 0/1
    "height_m": {"type": "continuous", "format": "mean_sd"},
    "baseline_weight_kg": {"type": "continuous", "format": "mean_sd"},
    "baseline_bmi": {"type": "continuous", "format": "mean_sd"},
    "baseline_fat_%": {"type": "continuous", "format": "mean_sd"},
    "baseline_muscle_%": {"type": "continuous", "format": "mean_sd"},
    "hunger_yn": {"type": "categorical", "format": "n_perc"}, # Binary 0/1
    "satiety_yn": {"type": "categorical", "format": "n_perc"}, # Binary 0/1
    "emotional_eating_yn": {"type": "categorical", "format": "n_perc"}, # Binary 0/1
    "emotional_eating_value_likert": {"type": "continuous", "format": "mean_sd"},
    "quantity_control_likert": {"type": "continuous", "format": "mean_sd"},
    "impulse_control_likert": {"type": "continuous", "format": "mean_sd"},
    "total_followup_days": {"type": "continuous", "format": "mean_sd"},
    "avg_days_between_measurements": {"type": "continuous", "format": "mean_sd"},
    "genomics_available": {"type": "categorical", "format": "n_perc"}, # Binary 0/1
    "total_wl_%": {"type": "continuous", "format": "mean_sd"},
    "bmi_reduction": {"type": "continuous", "format": "mean_sd"},
    "total_fat_loss_%": {"type": "continuous", "format": "mean_sd"},
    "total_muscle_change_%": {"type": "continuous", "format": "mean_sd"},
    "instant_dropout": {"type": "categorical", "format": "n_perc"}, # Binary 0/1
}

for w in time_windows:
    rows_config[f"{w}d_dropout"] = {"type": "categorical", "format": "n_perc"} # Binary 0/1
    rows_config[f"wl_{w}d_%"] = {"type": "continuous", "format": "mean_sd"}

for t in weight_loss_targets:
    rows_config[f"{t}%_wl_achieved"] = {"type": "categorical", "format": "n_perc"} # Binary 0/1
    rows_config[f"days_to_{t}%_wl"] = {"type": "continuous", "format": "mean_sd"}

# --- HELPER FUNCTIONS ---
def format_mean_sd(series):
    series_numeric = pd.to_numeric(series, errors="coerce").dropna()
    if len(series_numeric) < 1: return "N/A"
    return f"{series_numeric.mean():.2f} ± {series_numeric.std():.2f}"

def format_n_perc(series):
    series_cleaned = pd.to_numeric(series, errors="coerce").dropna() # Ensure numeric for sum
    if series_cleaned.empty: return "0 (0.0%)"
    n_positive = series_cleaned.astype(float).sum()
    total_valid = len(series_cleaned)
    percentage = (n_positive / total_valid) * 100 if total_valid > 0 else 0
    return f"{int(n_positive)} ({percentage:.1f}%)"

def format_p_value(p_val):
    if pd.isna(p_val): return "N/A"
    return "<0.001" if p_val < 0.001 else f"{p_val:.3f}"

def format_correlation_coeff(coeff):
    if pd.isna(coeff): return "N/A"
    return f"{coeff:.3f}"

def calculate_group_comparison_p_value(series1, series2, variable_type):
    series1_clean = pd.to_numeric(series1, errors="coerce").dropna()
    series2_clean = pd.to_numeric(series2, errors="coerce").dropna()

    if len(series1_clean) < 1 or len(series2_clean) < 1: return np.nan

    if variable_type == "continuous":
        if series1_clean.nunique() == 1 and series2_clean.nunique() == 1 and series1_clean.iloc[0] == series2_clean.iloc[0]: return 1.0
        if (series1_clean.std() == 0 and series2_clean.std() == 0 and series1_clean.iloc[0] != series2_clean.iloc[0]):
             if len(series1_clean) > 0 and len(series2_clean) > 0: return 0.0
        if len(series1_clean) < 2 and len(series2_clean) < 2:
            if series1_clean.nunique() == 1 and series2_clean.nunique() == 1 and series1_clean.iloc[0] == series2_clean.iloc[0]: return 1.0
            return np.nan
        _, p_val = stats.ttest_ind(series1_clean, series2_clean, equal_var=False, nan_policy='omit')
        return p_val
        
    elif variable_type == "categorical": # Assumes binary 0/1 data
        s1_success = series1_clean.sum()
        s1_total = len(series1_clean)
        s2_success = series2_clean.sum()
        s2_total = len(series2_clean)
        table = [[s1_success, s1_total - s1_success], [s2_success, s2_total - s2_success]]
        if np.sum(table) == 0 or min(s1_total, s2_total) == 0: return np.nan
        try:
            _, p_val = stats.fisher_exact(table)
        except ValueError:
            try:
                _, p_val, _, _ = stats.chi2_contingency(table)
            except ValueError: return np.nan
        return p_val
    return np.nan

def calculate_correlation_and_pvalue(outcome_series, cause_series, outcome_type):
    """Calculates correlation coefficient and its p-value."""
    # Ensure cause_series is numeric (0 or 1)
    # Ensure outcome_series is numeric (continuous or 0/1 for categorical)
    cause_s = pd.to_numeric(cause_series, errors='coerce')
    outcome_s = pd.to_numeric(outcome_series, errors='coerce')

    # Align series by index and drop NaNs from the pair
    aligned_df = pd.DataFrame({'outcome': outcome_s, 'cause': cause_s}).dropna()

    if len(aligned_df) < 5: # Minimum sample size for meaningful correlation
        return np.nan, np.nan
    if aligned_df['cause'].nunique() < 2: # Cause must be dichotomous (have at least two unique values, i.e. 0 and 1)
        return np.nan, np.nan
    if not aligned_df['cause'].isin([0, 1]).all(): # Cause must be 0/1
        # This should ideally be caught by preprocessing of cause columns
        print(f"Warning: Cause series for correlation is not strictly 0/1. Values: {aligned_df['cause'].unique()}")
        return np.nan, np.nan

    clean_outcome = aligned_df['outcome']
    clean_cause = aligned_df['cause']

    if outcome_type == "continuous":
        if clean_outcome.nunique() < 2: # Continuous outcome must have variance
            return np.nan, np.nan
        try:
            # Point-biserial correlation: x is dichotomous (cause), y is continuous (outcome)
            corr, p_val = stats.pointbiserialr(clean_cause, clean_outcome)
            return corr, p_val
        except Exception:
            return np.nan, np.nan
            
    elif outcome_type == "categorical": # Assuming binary outcome (0/1)
        if clean_outcome.nunique() < 2: # Categorical outcome must be dichotomous
            return np.nan, np.nan
        if not clean_outcome.isin([0, 1]).all(): # Outcome must be 0/1
            print(f"Warning: Categorical outcome for correlation is not strictly 0/1. Values: {clean_outcome.unique()}")
            return np.nan, np.nan
        try:
            # Phi coefficient from chi-squared
            contingency_table = pd.crosstab(clean_cause, clean_outcome)
            
            # Check if table is 2x2, if not, Phi is not directly applicable or needs adjustment
            if contingency_table.shape != (2,2):
                 # This can happen if, after filtering NaNs, one variable has only one category present.
                 return np.nan, np.nan

            chi2, p_val, _, _ = stats.chi2_contingency(contingency_table)
            n = contingency_table.sum().sum()
            if n == 0: return np.nan, np.nan
            
            phi = np.sqrt(chi2 / n)
            # Determine sign of phi based on table entries (ad - bc)
            # crosstab index: cause (0,1), columns: outcome (0,1)
            a = contingency_table.iloc[0,0] # Cause 0, Outcome 0
            b = contingency_table.iloc[0,1] # Cause 0, Outcome 1
            c = contingency_table.iloc[1,0] # Cause 1, Outcome 0
            d = contingency_table.iloc[1,1] # Cause 1, Outcome 1
            determinant = (a * d) - (b * c)
            if determinant < 0:
                phi = -phi
            return phi, p_val
        except Exception:
            return np.nan, np.nan
            
    return np.nan, np.nan

# --- MAIN ANALYSIS FUNCTION ---
def run_full_analysis():
    if not os.path.exists(DB_PATH_SURVIVAL):
        print(f"Survival database file not found: {DB_PATH_SURVIVAL}")
        return
    if not os.path.exists(DB_PATH_EMOTIONAL):
        print(f"Emotional eating database file not found: {DB_PATH_EMOTIONAL}")
        return

    print(f"Connecting to survival database: {DB_PATH_SURVIVAL}")
    conn_survival = sqlite3.connect(DB_PATH_SURVIVAL)
    print(f"Connecting to emotional eating database: {DB_PATH_EMOTIONAL}")
    conn_emotional = sqlite3.connect(DB_PATH_EMOTIONAL)

    print(f"Loading outcome data from '{SA_INPUT_TABLE}' in '{DB_PATH_SURVIVAL}'...")
    # df_outcomes will include 'medical_record_id' as per sa_input_table structure
    df_outcomes = pd.read_sql_query(f"SELECT * FROM \"{SA_INPUT_TABLE}\"", conn_survival)
    
    print(f"Loading cause data from '{MEDICAL_RECORDS_TABLE}' in '{DB_PATH_EMOTIONAL}'...")
    # Ensure medical_record_id is selected along with patient_id and cause columns
    quoted_cause_cols = [f'"{col}"' for col in WEIGHT_GAIN_CAUSE_COLUMNS]
    cause_cols_str = ", ".join(quoted_cause_cols)
    # MODIFICATION: Add medical_record_id to the query for df_causes
    df_causes_query = f"SELECT \"patient_id\", \"medical_record_id\", {cause_cols_str} FROM \"{MEDICAL_RECORDS_TABLE}\""
    df_causes = pd.read_sql_query(df_causes_query, conn_emotional)

    if df_outcomes.empty:
        print(f"Outcome table '{SA_INPUT_TABLE}' is empty. Aborting.")
        conn_survival.close()
        conn_emotional.close()
        return
    if df_causes.empty:
        print(f"Cause table '{MEDICAL_RECORDS_TABLE}' is empty. Aborting.")
        conn_survival.close()
        conn_emotional.close()
        return

    # Ensure patient_id and medical_record_id are of the same type for merging
    df_outcomes['patient_id'] = df_outcomes['patient_id'].astype(str)
    df_causes['patient_id'] = df_causes['patient_id'].astype(str)

    merge_on_columns = ["patient_id"]
    if 'medical_record_id' in df_outcomes.columns and 'medical_record_id' in df_causes.columns:
        df_outcomes['medical_record_id'] = df_outcomes['medical_record_id'].astype(str)
        df_causes['medical_record_id'] = df_causes['medical_record_id'].astype(str)
        merge_on_columns = ["patient_id", "medical_record_id"]
        print(f"Primary merge keys will be: {merge_on_columns}")
    else:
        print("Warning: 'medical_record_id' not found in both DataFrames. Merging on 'patient_id' only.")
        if 'medical_record_id' not in df_outcomes.columns:
            print(f"  'medical_record_id' missing in df_outcomes (from {SA_INPUT_TABLE})")
        if 'medical_record_id' not in df_causes.columns:
            print(f"  'medical_record_id' missing in df_causes (from {MEDICAL_RECORDS_TABLE})")

    print(f"Merging outcome and cause data using a LEFT join on {merge_on_columns}...")
    # MODIFICATION: Update merge to use appropriate columns
    merged_df = pd.merge(df_outcomes, df_causes, on=merge_on_columns, how="left")
    print(f"Shape of merged_df after merge: {merged_df.shape}")

    # Check for duplicates on merge_on_columns in merged_df, which would indicate issues if merge_on_columns should be unique
    if merged_df.duplicated(subset=merge_on_columns, keep=False).any():
        print(f"Warning: Duplicates found in merged_df based on {merge_on_columns} after merge. This might indicate an issue if these keys are expected to be unique.")
        # print(merged_df[merged_df.duplicated(subset=merge_on_columns, keep=False)].head())

    for cause_col in WEIGHT_GAIN_CAUSE_COLUMNS:
        if cause_col in merged_df.columns:
            merged_df[cause_col] = merged_df[cause_col].fillna(0).astype(int)
        else:
            print(f"Warning: Cause column '{cause_col}' not found. Will be skipped if in WEIGHT_GAIN_CAUSE_COLUMNS.")
            merged_df[cause_col] = 0 # Add as all zeros if missing, to prevent later KeyErrors if it was expected

    # Create 'any_cause_present'
    cause_columns_for_any_check = [col for col in WEIGHT_GAIN_CAUSE_COLUMNS if col in merged_df.columns and col != 'none_of_above'] # Exclude 'none_of_above'
    if cause_columns_for_any_check:
        merged_df['any_cause_present'] = merged_df[cause_columns_for_any_check].any(axis=1).astype(int)
    else:
        merged_df['any_cause_present'] = 0
        print("Warning: No valid cause columns found to create 'any_cause_present'. Defaulting to 0.")

    # Define 'instant_dropout'
    if 'nr_total_measurements' in merged_df.columns:
        merged_df['instant_dropout'] = (pd.to_numeric(merged_df['nr_total_measurements'], errors='coerce') == 1).astype(int)
    else:
        print("Warning: 'nr_total_measurements' column not found. 'instant_dropout' set to NA.")
        merged_df['instant_dropout'] = pd.NA

    # Define 'genomics_available'
    if 'genomics_sample_id' in merged_df.columns:
        merged_df['genomics_available'] = merged_df['genomics_sample_id'].notna().astype(int) & (merged_df['genomics_sample_id'] != '').astype(int)
    else:
        print("Warning: 'genomics_sample_id' column not found. 'genomics_available' set to NA.")
        merged_df['genomics_available'] = pd.NA

    # Ensure all categorical outcome variables in rows_config are 0/1 or NA
    for r_key, r_params in rows_config.items():
        if r_params['type'] == 'categorical' and r_key != "N": # N is special
            if r_key in merged_df.columns:
                # Convert to numeric, then to Int64 to allow NA and integers
                merged_df[r_key] = pd.to_numeric(merged_df[r_key], errors='coerce')
                # Check if values are other than 0, 1, NA. If so, it might be an issue for n_perc or correlations.
                # For now, we assume they are meant to be binary if 'categorical'.
                # If they are not 0/1, format_n_perc might misinterpret.
                # And categorical correlation expects 0/1.
                # A more robust step would be to explicitly map expected values to 0/1.
            else:
                 print(f"Warning: Categorical outcome variable '{r_key}' not found in merged_df.")


    # *** CORRECTED INITIALIZATION SCOPE FOR results_data ***
    results_data = [] 

    print("Starting analysis by cause...")
    comparison_group_columns = [col for col in WEIGHT_GAIN_CAUSE_COLUMNS if col in merged_df.columns] + ['any_cause_present']
    
    for row_key, row_params in rows_config.items():
        print(f"  Analyzing outcome variable: {row_key}")
        current_row_output = {'variable': row_key}
        outcome_variable_type = row_params['type']

        for cause_col_name in comparison_group_columns:
            if cause_col_name not in merged_df.columns: # Should not happen if 'any_cause_present' is created
                print(f"    Skipping cause group '{cause_col_name}' as it's not in the DataFrame.")
                stat_cause_val, stat_rest_val, p_val_group_comp_formatted = "Group Missing", "Group Missing", "N/A"
                corr_coeff_formatted, corr_p_val_formatted = "N/A", "N/A"
            elif row_key != "N" and row_key not in merged_df.columns:
                print(f"    Outcome variable '{row_key}' not found in DataFrame. Skipping stats for cause '{cause_col_name}'.")
                stat_cause_val, stat_rest_val, p_val_group_comp_formatted = "Outcome Missing", "Outcome Missing", "N/A"
                corr_coeff_formatted, corr_p_val_formatted = "N/A", "N/A"
            else:
                df_cause_group = merged_df[merged_df[cause_col_name] == 1]
                df_rest_group = merged_df[merged_df[cause_col_name] == 0]

                # Group comparison stats
                if row_key == "N":
                    stat_cause_val = str(len(df_cause_group))
                    stat_rest_val = str(len(df_rest_group))
                    p_val_group_comp_numeric = np.nan # No p-value for N comparison itself
                    corr_coeff, corr_p_val = np.nan, np.nan # No correlation for N
                else:
                    series_cause = df_cause_group[row_key] if not df_cause_group.empty else pd.Series(dtype='float64')
                    series_rest = df_rest_group[row_key] if not df_rest_group.empty else pd.Series(dtype='float64')

                    if outcome_variable_type == "continuous":
                        stat_cause_val = format_mean_sd(series_cause)
                        stat_rest_val = format_mean_sd(series_rest)
                    elif outcome_variable_type == "categorical": # Assumes 0/1
                        stat_cause_val = format_n_perc(series_cause)
                        stat_rest_val = format_n_perc(series_rest)
                    
                    p_val_group_comp_numeric = calculate_group_comparison_p_value(series_cause, series_rest, outcome_variable_type)
                    
                    # Correlation analysis
                    # outcome_series is merged_df[row_key], cause_series is merged_df[cause_col_name]
                    corr_coeff, corr_p_val = calculate_correlation_and_pvalue(merged_df[row_key], merged_df[cause_col_name], outcome_variable_type)

                p_val_group_comp_formatted = format_p_value(p_val_group_comp_numeric)
                corr_coeff_formatted = format_correlation_coeff(corr_coeff)
                corr_p_val_formatted = format_p_value(corr_p_val)

            current_row_output[f"{cause_col_name}_cause_stat"] = stat_cause_val
            current_row_output[f"{cause_col_name}_rest_stat"] = stat_rest_val
            current_row_output[f"{cause_col_name}_group_comp_pvalue"] = p_val_group_comp_formatted
            current_row_output[f"{cause_col_name}_corr_coeff"] = corr_coeff_formatted
            current_row_output[f"{cause_col_name}_corr_pvalue"] = corr_p_val_formatted
        
        results_data.append(current_row_output)

    results_df = pd.DataFrame(results_data)
    if not results_df.empty:
        results_df.set_index('variable', inplace=True)
        print(f"\nAnalysis complete. Saving results to table '{OUTPUT_TABLE}' in '{DB_PATH_SURVIVAL}'...")
        results_df.to_sql(OUTPUT_TABLE, conn_survival, if_exists="replace", index=True)
        print("Results saved successfully.")
        # print("\nPreview of results:")
        # print(results_df.head())
    else:
        print("No results were generated from the analysis.")

    if conn_survival: conn_survival.close(); print("Survival database connection closed.")
    if conn_emotional: conn_emotional.close(); print("Emotional eating database connection closed.")

# --- SCRIPT EXECUTION ---
if __name__ == "__main__":
    warnings.filterwarnings("ignore", category=RuntimeWarning, message="Degrees of freedom <= 0 for slice")
    warnings.filterwarnings("ignore", category=UserWarning, message="p-value may not be accurate for tables with counts < 5.")
    warnings.filterwarnings("ignore", category=RuntimeWarning, message="Mean of empty slice")
    warnings.filterwarnings("ignore", category=RuntimeWarning, message="invalid value encountered in scalar divide") # From stats.pointbiserialr with constant array
    warnings.filterwarnings("ignore", category=RuntimeWarning, message="Invalid value encountered in sqrt") # From phi if chi2/n is negative (should not happen) or n=0
    
    # Check if directories are defined (example check)
    if 'source_directory' not in globals() or 'wgcauses_directory' not in globals():
        print("ERROR: 'source_directory' or 'wgcauses_directory' is not defined.")
        print("Please define them at the beginning of the script.")
    else:
        run_full_analysis()


Connecting to survival database: C:\Users\Felhasználó\Desktop\Projects\PNK_DB2\paper1_emotional\wg_causes\survival_analysis.sqlite
Connecting to emotional eating database: C:\Users\Felhasználó\Desktop\Projects\PNK_DB2\DB2_standard\pnk_db2_filtered.sqlite
Loading outcome data from 'sa_input_table' in 'C:\Users\Felhasználó\Desktop\Projects\PNK_DB2\paper1_emotional\wg_causes\survival_analysis.sqlite'...
Loading cause data from 'medical_records_filtered' in 'C:\Users\Felhasználó\Desktop\Projects\PNK_DB2\DB2_standard\pnk_db2_filtered.sqlite'...
Primary merge keys will be: ['patient_id', 'medical_record_id']
Merging outcome and cause data using a LEFT join on ['patient_id', 'medical_record_id']...
Shape of merged_df after merge: (14223, 86)
Starting analysis by cause...
  Analyzing outcome variable: N
  Analyzing outcome variable: age
  Analyzing outcome variable: sex_f
  Analyzing outcome variable: height_m
  Analyzing outcome variable: baseline_weight_kg
  Analyzing outcome variable: basel

### Linear regressions for WL outcomes by WG drivers

### Linear Regressions for Weight Loss Outcomes by Weight Gain Drivers (Subset Analysis)

This script performs a series of linear regression analyses to investigate the association between various factors—including demographics, reported weight gain causes, and emotional eating metrics—and weight loss outcomes.

**1. Data Sources & Population:**
   - **Outcome & Primary Predictor Data:** Sourced from the `sa_input_table` within the `survival_analysis.sqlite` database (located in `wgcauses_directory`). This table provides variables like `patient_id`, `age`, `sex_f`, `emotional_eating_yn`, `emotional_eating_value_likert`, `total_wl_%`, and `total_followup_days`.
   - **Weight Gain Cause Data:** Sourced from the `medical_records_filtered` table within the `pnk_db2_filtered.sqlite` database (located in `source_directory`). This table provides `patient_id` and binary indicators for various `WEIGHT_GAIN_CAUSE_COLUMNS`.
   - **Analysis Subset:** The script performs an **inner join** between the outcome/predictor data and the cause data on `patient_id`. This means the entire analysis is conducted **only on the subset of patients for whom weight gain cause information is available** (approximately 2018 patients, as per prior context).

**2. Key Variables:**
   - **Dependent Variables:**
     - `wl_per_day_%`: Weight loss percentage per day of follow-up (derived).
     - `total_wl_%`: Total weight loss percentage.
   - **Independent Variables (Predictors):**
     - Demographics: `age`, `sex_f`.
     - Weight Gain Causes: A list of binary (0/1) variables from `WEIGHT_GAIN_CAUSE_COLUMNS`. Within the subset, any missing cause data is imputed to 0 (cause absent).
     - Emotional Eating: `emotional_eating_yn` (binary), `emotional_eating_value_likert` (continuous).

**3. Model Configurations:**
   The script iterates through the dependent variables and fits the following model configurations for each:
   - **Model 1: Base**
     - Predictors: `age`, `sex_f`, and all `WEIGHT_GAIN_CAUSE_COLUMNS`.
   - **Model 2: Base + Emotional Eating (Yes/No)**
     - Predictors: `age`, `sex_f`, all `WEIGHT_GAIN_CAUSE_COLUMNS`, and `emotional_eating_yn`.
   - **Model 3: Base + Emotional Eating (Likert Scale)**
     - Predictors: `age`, `sex_f`, all `WEIGHT_GAIN_CAUSE_COLUMNS`, and `emotional_eating_value_likert`.
   Predictors are included in the models in a predefined order (`MASTER_PREDICTOR_ORDER`).

**4. Handling of Missing Data within Models:**
   - After selecting the subset of ~2018 patients with known weight gain causes, the script further refines the dataset for each specific regression model.
   - Before fitting each model, `model_df_subset.dropna(inplace=True)` is called. This means that **only patients with complete, non-missing data for the dependent variable AND all independent variables included in that specific model configuration are used for that regression.**
   - Consequently:
     - The `Base` model will use the ~2018 patients, less any who have missing `age`, `sex_f`, or dependent variable data.
     - The `Base_plus_EmotionalEatingYN` model will use a further subset of these patients – only those who also have non-missing `emotional_eating_yn` data (expected N <= 583, potentially lower).
     - The `Base_plus_EmotionalEatingLikert` model will similarly use only those with non-missing `emotional_eating_value_likert` data (expected N <= 462, potentially lower).
   - The `N_Observations` reported for each model will reflect the actual sample size used for that specific regression after listwise deletion of missing values.

**5. Output:**
   - The results of all regression models (including coefficients, standard errors, t-statistics, p-values, confidence intervals, N, R-squared, and adjusted R-squared) are compiled into a single DataFrame.
   - This DataFrame is then saved as a new table named `linear_regression_wg_causes_results_v1_subset` in the `survival_analysis.sqlite` database, replacing the table if it already exists.

**Execution:**
   The script is intended to be run when `__name__ == "__main__"`, after ensuring the `source_directory` and `wgcauses_directory` global variables are correctly defined.

In [19]:
import os
import sqlite3
import pandas as pd
import numpy as np
import statsmodels.api as sm

# --- GLOBAL DIRECTORY DEFINITIONS (User needs to set these) ---
source_directory = r"C:\Users\Felhasználó\Desktop\Projects\PNK_DB2\DB2_standard"
wgcauses_directory = r"C:\Users\Felhasználó\Desktop\Projects\PNK_DB2\paper1_emotional\wg_causes"

# --- CONFIGURATION ---
DB_PATH_SURVIVAL = os.path.join(wgcauses_directory, "survival_analysis.sqlite")
DB_PATH_EMOTIONAL = os.path.join(source_directory, "pnk_db2_filtered.sqlite")
SA_INPUT_TABLE = "sa_input_table"
MEDICAL_RECORDS_TABLE = "medical_records_filtered"
OUTPUT_TABLE_REGRESSION = "linear_regression_wg_causes_results_v1_subset" # Modified output table name for clarity

WEIGHT_GAIN_CAUSE_COLUMNS = [
    'womens_health_and_pregnancy', 'mental_health', 'family_issues',
    'medication_disease_injury', 'physical_inactivity', 'eating_habits',
    'schedule', 'smoking_cessation', 'post_treatment_weight_regain',
    'pandemic', 'habits_and_circumstances', 'none_of_above',
]

SA_TABLE_PREDICTOR_OUTCOME_COLS = [
    'patient_id', 'age', 'sex_f', 'emotional_eating_yn',
    'emotional_eating_value_likert', 'total_wl_%', 'total_followup_days'
]


def run_linear_regressions_simplified():
    """
    Performs linear regression analyses for weight loss outcomes based on weight gain drivers.
    Modified to run on the subset of patients with available weight gain cause data.
    Models adjusted for separate emotional eating predictors and ordered IVs.
    """
    print("Starting linear regression analysis on subset with known causes...")

    # --- 1. Connect to Databases ---
    print(f"Connecting to survival database: {DB_PATH_SURVIVAL}")
    conn_survival = sqlite3.connect(DB_PATH_SURVIVAL)
    print(f"Connecting to emotional database: {DB_PATH_EMOTIONAL}")
    conn_emotional = sqlite3.connect(DB_PATH_EMOTIONAL)

    all_regression_results = []

    # --- 2. Load Data ---
    print(f"Loading outcome data from '{SA_INPUT_TABLE}' in '{DB_PATH_SURVIVAL}'...")
    df_outcomes = pd.read_sql_query(f"SELECT * FROM {SA_INPUT_TABLE}", conn_survival)
    
    print(f"Loading cause data from '{MEDICAL_RECORDS_TABLE}' in '{DB_PATH_EMOTIONAL}'...")
    # Ensure only patient_id, medical_record_id and cause columns are selected
    cause_columns_str = ', '.join(WEIGHT_GAIN_CAUSE_COLUMNS)
    # MODIFICATION: Add medical_record_id to the query for df_causes
    df_causes_query = f"SELECT patient_id, medical_record_id, {cause_columns_str} FROM {MEDICAL_RECORDS_TABLE}"
    df_causes = pd.read_sql_query(df_causes_query, conn_emotional)

    if df_outcomes.empty:
        print(f"Outcome table '{SA_INPUT_TABLE}' is empty. Aborting.")
        return
    if df_causes.empty:
        print(f"Cause table '{MEDICAL_RECORDS_TABLE}' is empty. Aborting.")
        return

    # Ensure patient_id is of the same type for merging
    df_outcomes['patient_id'] = df_outcomes['patient_id'].astype(str)
    df_causes['patient_id'] = df_causes['patient_id'].astype(str)
    
    # MODIFICATION: Ensure medical_record_id is present and of the same type for merging
    # This assumes 'medical_record_id' exists in df_outcomes. If not, this approach needs re-evaluation.
    if 'medical_record_id' in df_outcomes.columns and 'medical_record_id' in df_causes.columns:
        df_outcomes['medical_record_id'] = df_outcomes['medical_record_id'].astype(str)
        df_causes['medical_record_id'] = df_causes['medical_record_id'].astype(str)
        merge_on_columns = ["patient_id", "medical_record_id"]
    else:
        print("Warning: 'medical_record_id' not found in both DataFrames. Merging on 'patient_id' only.")
        print(f"df_outcomes columns: {df_outcomes.columns}")
        print(f"df_causes columns: {df_causes.columns}")
        merge_on_columns = ["patient_id"]

    # Merge outcome data with cause data
    print(f"Merging outcome and cause data on {merge_on_columns}...")
    # MODIFICATION: Update merge to use appropriate columns
    merged_df = pd.merge(df_outcomes, df_causes, on=merge_on_columns, how="left")
    print(f"Shape of merged_df: {merged_df.shape}")
    
    # --- 4. Preprocess Data (on the subset) ---
    print("Preprocessing merged subset data...")
    valid_cause_cols = []
    for col in WEIGHT_GAIN_CAUSE_COLUMNS:
        if col in merged_df.columns:
            # .fillna(0) here handles cases where a cause might be explicitly NaN within the 2018 known causes,
            # or if a column was all NaN in df_causes for the subset (less likely with inner join if df_causes is clean).
            merged_df[col] = pd.to_numeric(merged_df[col], errors='coerce').fillna(0).astype(int)
            valid_cause_cols.append(col)
        # else: # This case should be less frequent with an inner join if df_causes is well-formed
            # print(f"Warning: Cause column '{col}' not found in the subset after inner join.")


    cols_to_convert_numeric = ['age', 'sex_f', 'emotional_eating_yn',
                               'emotional_eating_value_likert', 'total_wl_%', 'total_followup_days']
    for col in cols_to_convert_numeric:
        if col in merged_df.columns:
            merged_df[col] = pd.to_numeric(merged_df[col], errors='coerce')
        # else:
            # print(f"Warning: Column '{col}' for numeric conversion not found in the subset.")


    if 'total_wl_%' in merged_df.columns and 'total_followup_days' in merged_df.columns:
        merged_df['total_followup_days_nozero'] = merged_df['total_followup_days'].replace(0, np.nan)
        merged_df['wl_per_day_%'] = merged_df['total_wl_%'] / merged_df['total_followup_days_nozero']
        merged_df['wl_per_day_%'].replace([np.inf, -np.inf], np.nan, inplace=True)
        merged_df.drop(columns=['total_followup_days_nozero'], inplace=True)
    else:
        merged_df['wl_per_day_%'] = np.nan
    print("Preprocessing complete for the subset.")

    # --- 5. Define Dependent Variables and Model Specifications ---
    dependent_vars_map = {
        'wl_per_day_perc': 'wl_per_day_%',
        'total_wl_perc': 'total_wl_%'
    }

    base_predictors_initial = ['age', 'sex_f']
    base_predictors = [p for p in base_predictors_initial if p in merged_df.columns] # Check against subset columns
    base_predictors.extend(valid_cause_cols) # valid_cause_cols are already confirmed to be in subset

    model_configurations = [
        {"name": "Base", "additional_predictors": []},
        {"name": "Base_plus_EmotionalEatingYN", "additional_predictors": ['emotional_eating_yn']},
        {"name": "Base_plus_EmotionalEatingLikert", "additional_predictors": ['emotional_eating_value_likert']}
    ]
    
    MASTER_PREDICTOR_ORDER = ['age', 'sex_f', 'emotional_eating_yn', 'emotional_eating_value_likert'] + WEIGHT_GAIN_CAUSE_COLUMNS

    # --- 6. Run Regressions (on the subset) ---
    print("\nStarting regression model fitting on the subset...")
    for dep_var_label, dep_var_col in dependent_vars_map.items():
        if dep_var_col not in merged_df.columns or merged_df[dep_var_col].isnull().all():
            print(f"Skipping DV '{dep_var_label}' as it's not in subset or all NaN.")
            continue

        print(f"\n--- Regressions for Dependent Variable: {dep_var_label} (column: {dep_var_col}) ---")

        for config in model_configurations:
            model_name = config["name"]
            
            valid_additional_predictors = []
            for add_pred in config["additional_predictors"]:
                if add_pred in merged_df.columns: # Check if additional predictor exists in the subset
                    valid_additional_predictors.append(add_pred)
                # else:
                    # print(f"Info: Additional predictor '{add_pred}' for model '{model_name}' not found in the subset.")

            potential_predictors_for_this_model_set = set(base_predictors + valid_additional_predictors)
            current_predictors = [
                p for p in MASTER_PREDICTOR_ORDER if p in potential_predictors_for_this_model_set
            ]

            if not current_predictors and not (len(potential_predictors_for_this_model_set) == 0 and model_name == "Base"): # Allow empty current_predictors if it's truly a base model with no valid base_predictors
                 if not (len(base_predictors)==0 and len(valid_additional_predictors)==0): # if base_predictors and valid_additional_predictors were empty, it's fine
                    print(f"  Skipping Model: {model_name} for DV {dep_var_label} - No valid predictors after ordering and checking presence in subset.")
                    continue


            print(f"  Running Model: {model_name}")
            print(f"    Predictors: {current_predictors}")

            cols_for_model_df = [dep_var_col] + [p for p in current_predictors if p in merged_df.columns]
            cols_for_model_df = list(dict.fromkeys(cols_for_model_df))

            model_df_subset = merged_df[cols_for_model_df].copy()
            model_df_subset.dropna(inplace=True) # THIS IS KEY FOR HANDLING MISSING EMOTIONAL EATING DATA

            actual_predictors_in_subset_df = [p for p in current_predictors if p in model_df_subset.columns]

            if not actual_predictors_in_subset_df:
                 print(f"    Skipping model '{model_name}' for '{dep_var_label}': No valid predictors remaining after NaN removal with DV.")
                 continue

            min_observations_needed = max(len(actual_predictors_in_subset_df) + 2, 20) # Min obs for model stability
            if model_df_subset.shape[0] < min_observations_needed:
                print(f"    Skipping model '{model_name}' for '{dep_var_label}': Insufficient observations ({model_df_subset.shape[0]}) after NaN removal. Needed {min_observations_needed}.")
                continue
            
            if model_df_subset[dep_var_col].nunique() < 2: # Dependent variable must have variance
                print(f"    Skipping model '{model_name}' for '{dep_var_label}': Dependent variable has no variance after NaN removal.")
                continue
            
            X = model_df_subset[actual_predictors_in_subset_df]
            y = model_df_subset[dep_var_col]
            
            constant_ivs = [iv for iv in actual_predictors_in_subset_df if X[iv].nunique() < 2]
            if constant_ivs:
                print(f"    Info: Model '{model_name}' for '{dep_var_label}' has constant IV(s) after filtering: {constant_ivs}.")
                # X = X.drop(columns=constant_ivs) # Optionally remove them
                # actual_predictors_in_subset_df = [p for p in actual_predictors_in_subset_df if p not in constant_ivs]
                # if not actual_predictors_in_subset_df:
                #     print(f"    Skipping model '{model_name}' for '{dep_var_label}': No predictors left after removing constant IVs.")
                #     continue
            
            if X.empty:
                print(f"    Skipping model '{model_name}' for '{dep_var_label}': No valid independent variables for OLS after processing.")
                continue

            X = sm.add_constant(X, has_constant='add')
            model = sm.OLS(y, X).fit()
            
            n_observations = model.nobs
            r_squared = model.rsquared
            adj_r_squared = model.rsquared_adj

            for i, predictor_name_in_model in enumerate(model.params.index):
                all_regression_results.append({
                    'Dependent_Variable': dep_var_label,
                    'Model_Name': model_name,
                    'Predictor': predictor_name_in_model,
                    'Coefficient': model.params[i],
                    'Std_Err': model.bse[i],
                    'T_Statistic': model.tvalues[i],
                    'P_Value': model.pvalues[i],
                    'Conf_Int_Lower': model.conf_int().iloc[i, 0],
                    'Conf_Int_Upper': model.conf_int().iloc[i, 1],
                    'N_Observations': n_observations,
                    'R_Squared': r_squared,
                    'Adj_R_Squared': adj_r_squared
                })
            print(f"    Model '{model_name}' for '{dep_var_label}' fitted successfully. N={int(n_observations)}, R2={r_squared:.3f}")

    # --- 7. Save Results ---
    if all_regression_results:
        results_df = pd.DataFrame(all_regression_results)
        print(f"\nSaving {len(results_df)} result rows to table '{OUTPUT_TABLE_REGRESSION}' in '{DB_PATH_SURVIVAL}'...")
        results_df.to_sql(OUTPUT_TABLE_REGRESSION, conn_survival, if_exists="replace", index=False)
        print("Regression results saved successfully.")
    else:
        print("\nNo regression results were generated to save.")

    # --- 8. Close Database Connections ---
    if conn_survival:
        conn_survival.close()
        print("Survival database connection closed.")
    if conn_emotional:
        conn_emotional.close()
        print("Emotional database connection closed.")
    
    print("Linear regression analysis on subset finished.")

# --- SCRIPT EXECUTION ---
if __name__ == "__main__":
    if 'source_directory' not in globals() or not source_directory or \
       'wgcauses_directory' not in globals() or not wgcauses_directory:
        print("ERROR: 'source_directory' or 'wgcauses_directory' is not properly defined at the top of the script.")
        print("Please define them before running.")
    else:
        run_linear_regressions_simplified()

Starting linear regression analysis on subset with known causes...
Connecting to survival database: C:\Users\Felhasználó\Desktop\Projects\PNK_DB2\paper1_emotional\wg_causes\survival_analysis.sqlite
Connecting to emotional database: C:\Users\Felhasználó\Desktop\Projects\PNK_DB2\DB2_standard\pnk_db2_filtered.sqlite
Loading outcome data from 'sa_input_table' in 'C:\Users\Felhasználó\Desktop\Projects\PNK_DB2\paper1_emotional\wg_causes\survival_analysis.sqlite'...
Loading cause data from 'medical_records_filtered' in 'C:\Users\Felhasználó\Desktop\Projects\PNK_DB2\DB2_standard\pnk_db2_filtered.sqlite'...
Merging outcome and cause data on ['patient_id', 'medical_record_id']...
Shape of merged_df: (14223, 86)
Preprocessing merged subset data...


C:\Users\Felhasználó\AppData\Local\Temp\ipykernel_17860\4135949937.py:112: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  merged_df['wl_per_day_%'].replace([np.inf, -np.inf], np.nan, inplace=True)
C:\Users\Felhasználó\AppData\Local\Temp\ipykernel_17860\4135949937.py:218: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  'Coefficient': model.params

Preprocessing complete for the subset.

Starting regression model fitting on the subset...

--- Regressions for Dependent Variable: wl_per_day_perc (column: wl_per_day_%) ---
  Running Model: Base
    Predictors: ['age', 'sex_f', 'womens_health_and_pregnancy', 'mental_health', 'family_issues', 'medication_disease_injury', 'physical_inactivity', 'eating_habits', 'schedule', 'smoking_cessation', 'post_treatment_weight_regain', 'pandemic', 'habits_and_circumstances', 'none_of_above']
    Model 'Base' for 'wl_per_day_perc' fitted successfully. N=14223, R2=0.020
  Running Model: Base_plus_EmotionalEatingYN
    Predictors: ['age', 'sex_f', 'emotional_eating_yn', 'womens_health_and_pregnancy', 'mental_health', 'family_issues', 'medication_disease_injury', 'physical_inactivity', 'eating_habits', 'schedule', 'smoking_cessation', 'post_treatment_weight_regain', 'pandemic', 'habits_and_circumstances', 'none_of_above']


C:\Users\Felhasználó\AppData\Local\Temp\ipykernel_17860\4135949937.py:218: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  'Coefficient': model.params[i],
C:\Users\Felhasználó\AppData\Local\Temp\ipykernel_17860\4135949937.py:219: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  'Std_Err': model.bse[i],
C:\Users\Felhasználó\AppData\Local\Temp\ipykernel_17860\4135949937.py:220: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  'T_Statistic': m

    Model 'Base_plus_EmotionalEatingYN' for 'wl_per_day_perc' fitted successfully. N=1645, R2=0.056
  Running Model: Base_plus_EmotionalEatingLikert
    Predictors: ['age', 'sex_f', 'emotional_eating_value_likert', 'womens_health_and_pregnancy', 'mental_health', 'family_issues', 'medication_disease_injury', 'physical_inactivity', 'eating_habits', 'schedule', 'smoking_cessation', 'post_treatment_weight_regain', 'pandemic', 'habits_and_circumstances', 'none_of_above']
    Model 'Base_plus_EmotionalEatingLikert' for 'wl_per_day_perc' fitted successfully. N=1456, R2=0.063

--- Regressions for Dependent Variable: total_wl_perc (column: total_wl_%) ---
  Running Model: Base
    Predictors: ['age', 'sex_f', 'womens_health_and_pregnancy', 'mental_health', 'family_issues', 'medication_disease_injury', 'physical_inactivity', 'eating_habits', 'schedule', 'smoking_cessation', 'post_treatment_weight_regain', 'pandemic', 'habits_and_circumstances', 'none_of_above']
    Model 'Base' for 'total_wl_per

C:\Users\Felhasználó\AppData\Local\Temp\ipykernel_17860\4135949937.py:218: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  'Coefficient': model.params[i],
C:\Users\Felhasználó\AppData\Local\Temp\ipykernel_17860\4135949937.py:219: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  'Std_Err': model.bse[i],
C:\Users\Felhasználó\AppData\Local\Temp\ipykernel_17860\4135949937.py:220: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  'T_Statistic': m

    Model 'Base_plus_EmotionalEatingYN' for 'total_wl_perc' fitted successfully. N=1645, R2=0.013
  Running Model: Base_plus_EmotionalEatingLikert
    Predictors: ['age', 'sex_f', 'emotional_eating_value_likert', 'womens_health_and_pregnancy', 'mental_health', 'family_issues', 'medication_disease_injury', 'physical_inactivity', 'eating_habits', 'schedule', 'smoking_cessation', 'post_treatment_weight_regain', 'pandemic', 'habits_and_circumstances', 'none_of_above']
    Model 'Base_plus_EmotionalEatingLikert' for 'total_wl_perc' fitted successfully. N=1456, R2=0.012

Saving 94 result rows to table 'linear_regression_wg_causes_results_v1_subset' in 'C:\Users\Felhasználó\Desktop\Projects\PNK_DB2\paper1_emotional\wg_causes\survival_analysis.sqlite'...
Regression results saved successfully.
Survival database connection closed.
Emotional database connection closed.
Linear regression analysis on subset finished.


## Weight gain causes

In [ ]:
import sqlite3
import pandas as pd

# Path to the SQLite database
db_path = r"C:\Users\Felhasználó\Desktop\Projects\PNK_DB2\DB2_standard\weight_gain_causes.sqlite"

# Connect and get table names
with sqlite3.connect(db_path) as conn:
    tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)
    print("Tables in database:", tables['name'].tolist())
    # Replace 'your_table_name' with the actual table name if known
    table_name = tables['name'].iloc[0]
    df_weight_gain_causes = pd.read_sql_query(f"SELECT * FROM [{table_name}]", conn)

df_weight_gain_causes

Tables in database: ['weight_gain_causes']


,patient_id,medical_record_id,weight_gain_cause,weight_gain_cause_en,womens_health_and_pregnancy,mental_health,family_issues,medication_disease_injury,physical_activity,eating_habits,schedule,smoking_cessation,post_treatment_weight_regain,pandemic,habits_and_circumstances,none_of_above
0,9897B34CEB969,13A04142A682D,"estrés, mucho trabajo","""stress, too much work""",0,1,0,0,0,0,1,0,0,0,1,0
1,9898644CEB969,13786742A682D,este último año ha cogido peso por tema mudanz...,"""this last year i have gained weight due to mo...",0,0,0,1,1,0,0,0,0,0,1,0
2,9899A04CEB969,10B67842A682D,EMBARAZO,"""pregnancy""",1,0,0,0,0,0,0,0,0,0,0,0
3,9899A94CEB969,12CE5B42A682D,FEZ MEDICAÇÃO PARA A DEPRESSÃO DURANTE A PANDE...,"""took medication for depression during the pan...",0,1,0,1,0,1,0,0,0,1,0,0
4,9899D04CEB969,13D74142A682D,HOUVE UM DESLEIXO ALIMENTAR NA QUANTIDADE,"""there was a neglect in food quantity""",0,0,0,0,0,1,0,0,0,0,1,0


## Misc

In [ ]:
import sqlite3

# Connect to the SQLite database
db_path = 'C:\\Users\\Felhasználó\\Desktop\\Projects\\PNK_DB2\\paper1_emotional\\survival_analysis.sqlite'
table_name = 'sa_input_table'

# Query to fetch unique values from the weight_gain_cause column
query = f"SELECT DISTINCT weight_gain_cause FROM {table_name}"

# Execute the query and fetch results
with sqlite3.connect(db_path) as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    results = cursor.fetchall()

# Extract and print the unique observations
unique_weight_gain_causes = [row[0] for row in results if row[0] is not None]
print("Unique observations in the weight_gain_cause column:")
for cause in unique_weight_gain_causes:
    print(cause)

Unique observations in the weight_gain_cause column:
PICOTEA EN EL RESTAURANTE SOLO HACE 1 COMIDA AL DIA. 
NO PODER MOVERSE POR DOLOR LUMBARES
ANSIEDAD POR PROBLEMAS LABORALES
DEPRESION
Estrés en el trabajo y horarios cambiantes 
No se cuida con cantidades. 
Estrés y poco ejercicio
SE QUEDÓ EMBARAZADA Y POR ESO LO DEJÓ
CAMBIO DE HÁBITOS, RUTINAS
PÍLDORA
QUARENTENA
malos hábitos alimentarios
COMER MAL
EMBARAZOS
DEPOIS DA PANDEMIA COMEÇOU A AUMENTAR PESO GRADUALMENTE. SEDENTARISMO.
JÁ FEZ LEV, DIETA 3 PASSOS, HERBALIFE, SEM RESULTADOS A MEDIO-LONGO PRAZO.
ESTAVA ACOMPANHADA (TRABALHO), ACHEI QUE NÃO ESTAVA CONFORTÁVEL NA PORMENORIZAÇÃO DESTE TEMA E NÃO O EXPLOREI MAIS.
desorden horario, come mas por las noches durante el dia se restringe
Tubo problemas de depresión en el pasado, ahora esta más estable y animada.
Problemas familiares y mucha ansiedad
No cuidarse, la vida social 
PANDEMIA
MALOS HÁBITOS, ANSIEDAD, MENOPAUSIA
Pandemia no se ha cuidado 
PROGRESIVO
COMENTA QUE SE SALE DE LA RU

In [ ]:
import sqlite3
import pandas as pd
import os

# Define the output Excel file path
DB_PATH = os.path.join(wgcauses_directory, "survival_analysis.sqlite")
output_excel_path = os.path.join(wgcauses_directory, 'survival_analysis.xlsx')

# DB_PATH = r"C:\Users\Felhasználó\Desktop\Projects\PNK_DB2\DB2_standard\pnk_db2_filtered.sqlite"
# output_excel_path = os.path.join(wgcauses_directory, 'filtered_db.xlsx')

# Connect to the SQLite database
with sqlite3.connect(DB_PATH) as conn:
    # Get the list of all tables in the database
    query = "SELECT name FROM sqlite_master WHERE type='table';"
    tables = pd.read_sql_query(query, conn)['name'].tolist()
    
    # Create a Pandas Excel writer
    with pd.ExcelWriter(output_excel_path, engine='openpyxl') as writer:
        # Loop through each table and save it as a sheet in the Excel file
        for table in tables:
            df = pd.read_sql_query(f"SELECT * FROM {table}", conn)
            df.to_excel(writer, sheet_name=table, index=False)

print(f"Database exported to Excel at: {output_excel_path}")

c:\Users\Felhasználó\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\workbook\child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")


Database exported to Excel at: C:\Users\Felhasználó\Desktop\Projects\PNK_DB2\paper1_emotional\wg_causes\survival_analysis.xlsx
